In [4]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import tensorflow_decision_forests as tfdf
import pandas as pd

https://www.tensorflow.org/decision_forests/api_docs/python/tfdf/keras/RandomForestModel

In [2]:
predictors = ["grid", "position", "pos_delta", "driver_code", "constructor_code", "circuit_code", "grid_rolling", "position_rolling", "pos_delta_rolling"]

In [5]:
data = pd.read_csv("./data/final_rolling.csv")

In [6]:
data.dropna(subset="position", inplace=True)

In [7]:
data["position"] = data["position"].astype("int")

In [8]:
training = data[data["year"] < 2022]
test = data[data["year"] >= 2022]

In [9]:
train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(training[predictors], label="position")
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(test[predictors])

In [10]:
model = tfdf.keras.RandomForestModel(verbose=0)

model.fit(train_ds)

[INFO 24-08-06 16:52:07.0378 CEST kernel.cc:1233] Loading model from path /var/folders/hk/ffpx1y0s2cn133v15g3yks080000gn/T/tmplnnmi442/model/ with prefix e12d508573264f9c
[INFO 24-08-06 16:52:08.0418 CEST decision_forest.cc:734] Model loaded with 300 root(s), 745304 node(s), and 8 input feature(s).
[INFO 24-08-06 16:52:08.0419 CEST abstract_model.cc:1344] Engine "RandomForestGeneric" built
[INFO 24-08-06 16:52:08.0419 CEST kernel.cc:1061] Use fast generic engine


In [11]:
predictions = model.predict(test_ds)

2/2 [==============================] - 0s 3ms/step


### Prediction tables:
    Rows - prediction number
    Column - driverId

In [14]:
predictions_df = pd.DataFrame(predictions)

In [19]:
full_table = pd.merge(test, predictions_df, on=test.index)

full_table

,key_0,raceId,grid,position,year,date,time,circuitRef,driverRef,constructorRef,...,24,25,26,27,28,29,30,31,32,33
0,25400,1074,1.0,1,2022,2022-03-20,15:00:00,bahrain,leclerc,ferrari,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,25401,1074,3.0,2,2022,2022-03-20,15:00:00,bahrain,sainz,ferrari,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2,25402,1074,5.0,3,2022,2022-03-20,15:00:00,bahrain,hamilton,mercedes,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,25403,1074,9.0,4,2022,2022-03-20,15:00:00,bahrain,russell,mercedes,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,25404,1074,7.0,5,2022,2022-03-20,15:00:00,bahrain,kevin_magnussen,haas,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1013,26554,1134,20.0,16,2024,2024-07-28,15:00,spa,tsunoda,rb,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1014,26555,1134,18.0,17,2024,2024-07-28,15:00,spa,sargeant,williams,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1015,26556,1134,16.0,18,2024,2024-07-28,15:00,spa,hulkenberg,haas,...,0.0,0.01,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1016,26557,1134,19.0,19,2024,2024-07-28,15:00,spa,zhou,sauber,...,0.0,0.00,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


In [24]:
evaluation = model.make_inspector().evaluation()

In [25]:
evaluation.accuracy

0.9490179976733046

In [21]:
preds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33]

In [22]:
full_table["max_pred"] = full_table[preds].max(axis=1)

In [23]:
full_table[["raceId", "driverRef", "constructorRef", "position", "circuitRef", "max_pred"]]

,raceId,driverRef,constructorRef,position,circuitRef,max_pred
0,1074,leclerc,ferrari,1,bahrain,0.973333
1,1074,sainz,ferrari,2,bahrain,0.899999
2,1074,hamilton,mercedes,3,bahrain,0.743333
3,1074,russell,mercedes,4,bahrain,0.620000
4,1074,kevin_magnussen,haas,5,bahrain,0.486666
...,...,...,...,...,...,...
1013,1134,tsunoda,rb,16,spa,0.620000
1014,1134,sargeant,williams,17,spa,0.573333
1015,1134,hulkenberg,haas,18,spa,0.273333
1016,1134,zhou,sauber,19,spa,0.423333
